# Multi-Source Cultural Salience Analysis

Combine three independent measures of cultural salience:
1. **Jeopardy frequency** — what quiz-show writers consider worth asking about (1984–2024)
2. **Wikipedia pageviews** — what the public actually reads about (2020–2024)
3. **Crossword frequency** — what puzzle constructors assume solvers know (42 publications)

Plus a benchmark comparison against **Hirsch’s Cultural Literacy Dictionary** (~6,900 entries).

Questions:
1. How correlated are these three signals? Do they capture the same construct?
2. Which concepts rank high on all three? (Consensus cultural knowledge)
3. Which concepts diverge? (Domain-specific salience)
4. Can a composite score improve on any single source?

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

In [ ]:
# Load all data sources
jeopardy = pd.read_csv("../data/jeopardy_concepts_v3.csv", index_col=0,
                        engine="python", on_bad_lines="skip")
pageviews = pd.read_csv("../data/wikipedia_pageviews.csv")
crossword = pd.read_csv("../data/xd_crossword_answers.csv")
hirsch = pd.read_csv("../data/hirsch_cultural_literacy_terms.csv")

print(f"Jeopardy concepts: {len(jeopardy):,}")
print(f"Wikipedia pageview records: {len(pageviews):,} ({(pageviews['total_views']>0).sum():,} with data)")
print(f"Crossword unique answers: {len(crossword):,}")
print(f"Hirsch dictionary terms: {len(hirsch):,}")

## 1. Merge Data Sources

Start from the Jeopardy concept list (our primary dataset) and join Wikipedia pageviews (via Wikidata QID) and crossword frequency (via normalized text matching).

In [ ]:
# --- Wikipedia pageviews: merge on concept name ---
pv = pageviews[pageviews["total_views"] > 0][["concept", "wiki_title", "total_views", "monthly_avg"]].copy()
pv = pv.rename(columns={"total_views": "wiki_views", "monthly_avg": "wiki_monthly_avg"})

# Merge on concept (from wikidata_linking, which shares concept names with jeopardy)
df = jeopardy.copy()
df = df.merge(pv, on="concept", how="left")

# --- Crossword frequency: merge on normalized text ---
def xw_normalize(text):
    """Normalize text to match crossword answer format (uppercase, no spaces/punctuation)."""
    if not isinstance(text, str):
        return ""
    return text.upper().replace(" ", "").replace("_", "").replace("-", "").replace(".", "").replace("'", "")

df["xw_key"] = df["concept"].apply(xw_normalize)
df["xw_key2"] = df["display_name"].apply(xw_normalize)

xw_lookup = crossword.set_index(
    crossword["answer"].str.upper().str.replace(" ", "").str.replace("_", "")
    .str.replace("-", "").str.replace(".", "").str.replace("'", "")
)["frequency"].to_dict()

df["xw_frequency"] = df["xw_key"].map(xw_lookup).fillna(df["xw_key2"].map(xw_lookup)).fillna(0).astype(int)

# Summary
has_wiki = df["wiki_views"].notna().sum()
has_xw = (df["xw_frequency"] > 0).sum()
has_both = ((df["wiki_views"].notna()) & (df["xw_frequency"] > 0)).sum()
has_all3 = ((df["wiki_views"].notna()) & (df["xw_frequency"] > 0) & (df["frequency"] >= 1)).sum()

print(f"Merged dataset: {len(df):,} concepts")
print(f"  With Jeopardy data: {len(df):,} (all)")
print(f"  With Wikipedia pageviews: {has_wiki:,}")
print(f"  With crossword frequency: {has_xw:,}")
print(f"  With all three signals: {has_all3:,}")

# Drop temp columns
df.drop(columns=["xw_key", "xw_key2"], inplace=True)

## 2. Correlation Between Signals

For concepts with all three signals, how correlated are they? High correlation means they capture the same construct; low correlation means each adds unique information.

In [ ]:
# Focus on concepts with all three signals and meaningful Jeopardy frequency
trio = df[(df["wiki_views"].notna()) & (df["xw_frequency"] > 0) & (df["frequency"] >= 3)].copy()
print(f"Concepts with all three signals (Jeopardy freq >= 3): {len(trio):,}")

# Log-transform for correlation (all are heavy-tailed)
trio["log_jeopardy"] = np.log1p(trio["frequency"])
trio["log_wiki"] = np.log1p(trio["wiki_views"])
trio["log_xw"] = np.log1p(trio["xw_frequency"])

# Spearman rank correlations (more robust for heavy-tailed distributions)
pairs = [
    ("Jeopardy vs Wikipedia", "log_jeopardy", "log_wiki"),
    ("Jeopardy vs Crossword", "log_jeopardy", "log_xw"),
    ("Wikipedia vs Crossword", "log_wiki", "log_xw"),
]

print(f"\nSpearman rank correlations (n={len(trio):,}):")
for label, col1, col2 in pairs:
    rho, pval = stats.spearmanr(trio[col1], trio[col2])
    print(f"  {label:30s}  rho = {rho:.3f}  (p = {pval:.2e})")

# Scatter matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (label, col1, col2) in zip(axes, pairs):
    ax.scatter(trio[col1], trio[col2], alpha=0.15, s=8)
    rho, _ = stats.spearmanr(trio[col1], trio[col2])
    ax.set_xlabel(col1.replace("log_", "").title() + " (log)")
    ax.set_ylabel(col2.replace("log_", "").title() + " (log)")
    ax.set_title(f"{label}\n(Spearman rho = {rho:.3f})")
fig.suptitle("Pairwise Correlations Between Cultural Salience Signals", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../data/signal_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Composite Cultural Salience Score

Create a composite score by percentile-ranking each signal and averaging. This is more robust than raw values because each source has different scales and distributions.

In [ ]:
# Build composite on ALL concepts (not just trio)
# Percentile rank within each signal (0-100), NaN for missing
df["pctile_jeopardy"] = df["frequency"].rank(pct=True) * 100
df["pctile_wiki"] = df["wiki_views"].rank(pct=True) * 100
df["pctile_xw"] = df["xw_frequency"].replace(0, np.nan).rank(pct=True) * 100

# Composite: mean of available percentiles
pctile_cols = ["pctile_jeopardy", "pctile_wiki", "pctile_xw"]
df["n_signals"] = df[pctile_cols].notna().sum(axis=1)
df["composite_score"] = df[pctile_cols].mean(axis=1)

print(f"Composite score coverage:")
print(df["n_signals"].value_counts().sort_index().to_string())
print(f"\nComposite score distribution:")
print(df["composite_score"].describe().to_string())

# Top concepts by composite score (require all 3 signals)
top_composite = df[df["n_signals"] == 3].nlargest(50, "composite_score")
print(f"\nTop 50 concepts by composite cultural salience (all 3 signals):")
print(top_composite[["concept", "display_name", "entity_type", "frequency",
                      "wiki_views", "xw_frequency", "composite_score"]].to_string())

## 4. Signal Divergence: Where Sources Disagree

Identify concepts that rank much higher on one signal than others — these reveal the biases of each source.

In [ ]:
# Focus on concepts with all 3 signals
full = df[df["n_signals"] == 3].copy()

# Divergence: max percentile - min percentile
full["pctile_range"] = full[pctile_cols].max(axis=1) - full[pctile_cols].min(axis=1)

# High on Jeopardy, low elsewhere
full["jeopardy_excess"] = full["pctile_jeopardy"] - full[["pctile_wiki", "pctile_xw"]].mean(axis=1)
# High on Wikipedia, low elsewhere
full["wiki_excess"] = full["pctile_wiki"] - full[["pctile_jeopardy", "pctile_xw"]].mean(axis=1)
# High on crossword, low elsewhere
full["xw_excess"] = full["pctile_xw"] - full[["pctile_jeopardy", "pctile_wiki"]].mean(axis=1)

cols = ["concept", "display_name", "entity_type", "frequency", "wiki_views",
        "xw_frequency", "pctile_jeopardy", "pctile_wiki", "pctile_xw"]

print("=== JEOPARDY-BIASED (high on Jeopardy, low on Wiki + Crossword) ===")
print(full.nlargest(20, "jeopardy_excess")[cols].to_string())

print("\n=== WIKIPEDIA-BIASED (high on Wiki, low on Jeopardy + Crossword) ===")
print(full.nlargest(20, "wiki_excess")[cols].to_string())

print("\n=== CROSSWORD-BIASED (high on Crossword, low on Jeopardy + Wiki) ===")
print(full.nlargest(20, "xw_excess")[cols].to_string())

## 5. Hirsch Coverage by Composite Score

How well does the composite score predict whether a concept appears in Hirsch's canon?

In [ ]:
import re

def normalize(text):
    if not isinstance(text, str):
        return ""
    s = text.lower().strip()
    s = re.sub(r"^(the|a|an)\s+", "", s)
    s = re.sub(r"[,;:!?'\"()\[\]]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Build Hirsch lookup
hirsch_norms = set(hirsch["term"].apply(normalize))
hirsch_flipped = set()
for term in hirsch["term"]:
    if ", " in str(term):
        parts = term.split(", ", 1)
        if len(parts) == 2 and parts[1][0:1].isupper():
            hirsch_flipped.add(normalize(parts[1] + " " + parts[0]))
hirsch_all = hirsch_norms | hirsch_flipped

df["in_hirsch"] = (
    df["concept"].apply(normalize).isin(hirsch_all) |
    df["display_name"].apply(normalize).isin(hirsch_all)
)

# Coverage by composite score decile
scored = df[df["n_signals"] >= 2].copy()
scored["score_decile"] = pd.qcut(scored["composite_score"], 10, labels=False, duplicates="drop") + 1

decile_cov = scored.groupby("score_decile").agg(
    n=("concept", "size"),
    in_hirsch=("in_hirsch", "sum"),
).reset_index()
decile_cov["pct"] = 100 * decile_cov["in_hirsch"] / decile_cov["n"]

print("Hirsch coverage by composite score decile:")
print(decile_cov.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(decile_cov["score_decile"], decile_cov["pct"])
ax.set_xlabel("Composite Score Decile (1=lowest, 10=highest)")
ax.set_ylabel("% in Hirsch Dictionary")
ax.set_title("Hirsch Dictionary Coverage by Multi-Source Composite Score")
for x, y in zip(decile_cov["score_decile"], decile_cov["pct"]):
    ax.text(x, y + 0.5, f"{y:.0f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("../data/hirsch_by_composite.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Entity Type Breakdown

How do the three signals weight different types of cultural knowledge?

In [ ]:
# Median percentile by entity type for each signal
type_signals = df[df["n_signals"] == 3].groupby("entity_type").agg(
    n=("concept", "size"),
    med_jeopardy=("pctile_jeopardy", "median"),
    med_wiki=("pctile_wiki", "median"),
    med_xw=("pctile_xw", "median"),
    med_composite=("composite_score", "median"),
).reset_index()
type_signals = type_signals[type_signals["n"] >= 20].sort_values("med_composite", ascending=False)

print("Median percentile rank by entity type (all 3 signals):")
print(type_signals.to_string(index=False, float_format="{:.1f}".format))

# Grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(type_signals))
w = 0.25
ax.bar(x - w, type_signals["med_jeopardy"], w, label="Jeopardy")
ax.bar(x, type_signals["med_wiki"], w, label="Wikipedia")
ax.bar(x + w, type_signals["med_xw"], w, label="Crossword")
ax.set_xticks(x)
ax.set_xticklabels(type_signals["entity_type"], rotation=45, ha="right")
ax.set_ylabel("Median Percentile Rank")
ax.set_title("Signal Strength by Entity Type")
ax.legend()
plt.tight_layout()
plt.savefig("../data/entity_type_signals.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary & Export

In [ ]:
# Export enriched dataset
export_cols = ["concept", "display_name", "entity_type", "frequency", "seasons_appeared",
               "category_breadth", "importance", "wiki_title", "wiki_views",
               "wiki_monthly_avg", "xw_frequency", "pctile_jeopardy", "pctile_wiki",
               "pctile_xw", "n_signals", "composite_score", "in_hirsch"]
export = df[[c for c in export_cols if c in df.columns]].copy()
export = export.sort_values("composite_score", ascending=False)
export.to_csv("../data/cultural_salience_composite.csv", index=False)
print(f"Exported {len(export):,} concepts to data/cultural_salience_composite.csv")

# Print summary
print(f"\n{'=' * 65}")
print(f"MULTI-SOURCE CULTURAL SALIENCE SUMMARY")
print(f"{'=' * 65}")
print(f"\nData sources:")
print(f"  Jeopardy! answers (1984-2024): {len(df):,} concepts")
print(f"  Wikipedia pageviews (2020-2024): {has_wiki:,} linked")
print(f"  Crossword answers (42 pubs): {has_xw:,} matched")
print(f"  All three signals: {has_all3:,} concepts")

# How many Hirsch entries covered?
in_h = df["in_hirsch"].sum()
print(f"\nHirsch dictionary:")
print(f"  Hirsch entries in our data: ~{in_h:,}")
print(f"  Top-decile Hirsch coverage: {decile_cov.iloc[-1]['pct']:.0f}%")

# Correlation summary
for label, col1, col2 in pairs:
    rho, _ = stats.spearmanr(trio[col1], trio[col2])
    print(f"  {label}: rho = {rho:.3f}")